In [1]:
from __future__ import annotations

import asyncio
import math
import os
import sys
from pathlib import Path

BACKEND_DIR = Path.cwd().resolve()
if not (BACKEND_DIR / 'app').is_dir():
    BACKEND_DIR = BACKEND_DIR.parent

if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from dotenv import load_dotenv

load_dotenv(BACKEND_DIR / '.env')

assert os.environ.get('GOOGLE_API_KEY'), (
    'Set GOOGLE_API_KEY in backend/.env before running this notebook.'
)

print(f'Backend directory: {BACKEND_DIR}')
print('GOOGLE_API_KEY found.')

Backend directory: /home/pranavsood/Documents/Projects/EduTube-AI/backend
GOOGLE_API_KEY found.


In [2]:
from langchain_core.documents import Document

from app.services.rag.store import VectorStore

store = VectorStore()
print(f'Model: {store.embedding_model.model}')

/home/pranavsood/Documents/Projects/EduTube-AI/backend/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Model: models/gemini-embedding-2


In [4]:
documents = [
    Document(page_content='जड़त्व किसी वस्तु की गति में परिवर्तन के प्रति उसका प्रतिरोध है।'),
    Document(page_content='प्रकाश संश्लेषण पौधों में प्रकाश ऊर्जा को रासायनिक ऊर्जा में परिवर्तित करता है।'),
    Document(page_content='न्यूटन का दूसरा नियम बल, द्रव्यमान और त्वरण के बीच संबंध बताता है।'),
]

embeddings = await store.embed_docs(documents)

assert len(embeddings) == len(documents)
assert len({len(embedding) for embedding in embeddings}) == 1
assert all(math.isfinite(value) for embedding in embeddings for value in embedding)

dimension = len(embeddings[0])
print(f'Embedded {len(embeddings)} documents; vector dimension: {dimension}')

Embedded 3 documents; vector dimension: 3072


In [5]:
query = 'Which physics principle explains why a stationary object stays still?'
query_embedding = await store.embedding_model.aembed_query(query)

assert len(query_embedding) == dimension
assert all(math.isfinite(value) for value in query_embedding)

def cosine_similarity(left: list[float], right: list[float]) -> float:
    numerator = sum(a * b for a, b in zip(left, right))
    left_norm = math.sqrt(sum(a * a for a in left))
    right_norm = math.sqrt(sum(b * b for b in right))
    return numerator / (left_norm * right_norm)

ranked_results = sorted(
    (
        (cosine_similarity(query_embedding, embedding), document.page_content)
        for document, embedding in zip(documents, embeddings)
    ),
    reverse=True,
)

for score, text in ranked_results:
    print(f'{score:.4f}  {text}')

assert ranked_results[0][1] == documents[0].page_content
print('Embedding smoke test passed.')

0.6967  जड़त्व किसी वस्तु की गति में परिवर्तन के प्रति उसका प्रतिरोध है।
0.5877  न्यूटन का दूसरा नियम बल, द्रव्यमान और त्वरण के बीच संबंध बताता है।
0.4768  प्रकाश संश्लेषण पौधों में प्रकाश ऊर्जा को रासायनिक ऊर्जा में परिवर्तित करता है।
Embedding smoke test passed.


In [7]:
query_hn = 'भौतिकी का कौन सा सिद्धांत यह बताता है कि कोई स्थिर वस्तु स्थिर क्यों रहती है?'
query_embedding_hn = await store.embedding_model.aembed_query(query_hn)

assert len(query_embedding_hn) == dimension
assert all(math.isfinite(value) for value in query_embedding_hn)

ranked_results = sorted(
    (
        (cosine_similarity(query_embedding_hn, embedding), document.page_content)
        for document, embedding in zip(documents, embeddings)
    ),
    reverse=True,
)

for score, text in ranked_results:
    print(f'{score:.4f}  {text}')

assert ranked_results[0][1] == documents[0].page_content
print('Embedding smoke test passed.')

0.7405  जड़त्व किसी वस्तु की गति में परिवर्तन के प्रति उसका प्रतिरोध है।
0.6735  न्यूटन का दूसरा नियम बल, द्रव्यमान और त्वरण के बीच संबंध बताता है।
0.5256  प्रकाश संश्लेषण पौधों में प्रकाश ऊर्जा को रासायनिक ऊर्जा में परिवर्तित करता है।
Embedding smoke test passed.
